# Training models in one collection date to infer on others

## Loading the Database

In [1]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
import numpy as np

In [2]:
IMG_SIZE = 512
BATCH_SIZE = 8
NUM_CLASSES = 1
CROP = "corn"  # "wheat", "sorghum" or "corn"
COLLECTION_DATE = "6"

dataset_path = f"../data/{CROP}/"

class SegmentationDataset(Dataset):
    def __init__(self, root_dir, train=True, collection_dates=None):
        self.root_dir = root_dir
        self.train = train
        self.img_dir = os.path.join(root_dir, "train/images" if train else "test/images")
        self.mask_dir = os.path.join(root_dir, "train/masks" if train else "test/masks")

        # List all image files
        image_files = sorted([
            f for f in os.listdir(self.img_dir)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])
        mask_files = sorted([
            f for f in os.listdir(self.mask_dir)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])
        
        # Filter by collection date if provided
        if collection_dates is not None:
            # Accept single date or list of dates
            if isinstance(collection_dates, (str, int)):
                collection_dates = [str(collection_dates)]
            image_files = [f for f in image_files if f.split('_')[0] in collection_dates]
            mask_files = [f for f in mask_files if f.split('_')[0] in collection_dates]
        print(image_files)
        # Ensure matching images/masks
        assert len(image_files) == len(mask_files), \
            f"Image/Mask count mismatch: {len(image_files)} vs {len(mask_files)}"

        self.image_files = image_files
        self.mask_files = mask_files

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.image_files[idx])
        mask_path = os.path.join(self.mask_dir, self.mask_files[idx])

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")  # grayscale mask

        image = TF.to_tensor(image)
        image = TF.normalize(image, mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        mask = torch.from_numpy(np.array(mask)).float().unsqueeze(0)
        mask = (mask > 0).float()

        return image, mask

train_dataset = SegmentationDataset(dataset_path, train=True, collection_dates=COLLECTION_DATE)
test_dataset = SegmentationDataset(dataset_path, train=False, collection_dates=COLLECTION_DATE)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"✅ Train size: {len(train_dataset)} | Test size: {len(test_dataset)}")

['6_100_1.jpg', '6_100_1_aug1.jpg', '6_100_1_aug2.jpg', '6_100_1_aug3.jpg', '6_100_1_aug4.jpg', '6_100_1_aug5.jpg', '6_100_2.jpg', '6_100_2_aug1.jpg', '6_100_2_aug2.jpg', '6_100_2_aug3.jpg', '6_100_2_aug4.jpg', '6_100_2_aug5.jpg', '6_100_3.jpg', '6_100_3_aug1.jpg', '6_100_3_aug2.jpg', '6_100_3_aug3.jpg', '6_100_3_aug4.jpg', '6_100_3_aug5.jpg', '6_101_1.jpg', '6_101_1_aug1.jpg', '6_101_1_aug2.jpg', '6_101_1_aug3.jpg', '6_101_1_aug4.jpg', '6_101_1_aug5.jpg', '6_101_2.jpg', '6_101_2_aug1.jpg', '6_101_2_aug2.jpg', '6_101_2_aug3.jpg', '6_101_2_aug4.jpg', '6_101_2_aug5.jpg', '6_101_3.jpg', '6_101_3_aug1.jpg', '6_101_3_aug2.jpg', '6_101_3_aug3.jpg', '6_101_3_aug4.jpg', '6_101_3_aug5.jpg', '6_102_1.jpg', '6_102_1_aug1.jpg', '6_102_1_aug2.jpg', '6_102_1_aug3.jpg', '6_102_1_aug4.jpg', '6_102_1_aug5.jpg', '6_102_2.jpg', '6_102_2_aug1.jpg', '6_102_2_aug2.jpg', '6_102_2_aug3.jpg', '6_102_2_aug4.jpg', '6_102_2_aug5.jpg', '6_102_3.jpg', '6_102_3_aug1.jpg', '6_102_3_aug2.jpg', '6_102_3_aug3.jpg', '6_1

In [3]:
for i in range(3):
    img, mask = train_dataset[i]
    print(img.shape, mask.shape)

torch.Size([3, 512, 512]) torch.Size([1, 512, 512])
torch.Size([3, 512, 512]) torch.Size([1, 512, 512])
torch.Size([3, 512, 512]) torch.Size([1, 512, 512])


## Importing Models

In [3]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


### U-NET

In [4]:
import sys
sys.path.append('../')
from utils.models.uNet import UNet

channels = [32, 64, 128, 256, 512]
model = UNet(in_channels=3, out_channels=1, channels=channels, bilinear=True, use_batchnorm=True)
model.to(device)
modelName = "U-NET"

### SegFormer

In [4]:
import sys
sys.path.append('../')
from utils.models.SegFormer import segformer

model = segformer(in_channels = 3, num_classes = 1)
model.to(device)
modelName = "SegFormer"

c:\Users\gnoceras\Documents\SegmentationStudy\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\gnoceras\Documents\SegmentationStudy\.venv\lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


## Training

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import os
from tqdm import tqdm

In [6]:
# --- Configuration ---
EPOCHS = 200
early_stopping_patience = 5
best_val_loss = float('inf') 
patience_counter = 0
scaler = torch.amp.GradScaler('cuda')

# --- Loss & Optimizer ---
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
lr_scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3)

checkpoint_path = os.path.join("../models/", CROP + "_" + modelName + "_date" + COLLECTION_DATE + "_seg.pt")
print(f"Model checkpoints will be saved to: {checkpoint_path}")

# --- Tracking ---
train_losses, val_losses = [], []

Model checkpoints will be saved to: ../models/corn_SegFormer_date6_seg.pt


In [7]:
# --- Training Loop ---
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    with tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", unit="batch") as tepoch:
        for inputs, labels in tepoch:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                outputs = model(inputs)
                if torch.isnan(outputs).any() or torch.isinf(outputs).any():
                    print("NaN or Inf in model outputs!")
                if torch.isnan(labels).any() or torch.isinf(labels).any():
                    print("NaN or Inf in labels!")
                if labels.sum() == 0:
                    print("Skipping batch with all empty masks")
                    continue
                loss = loss_fn(outputs, labels)
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()
            tepoch.set_postfix(loss=running_loss / (len(tepoch)))
    avg_train_loss = running_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # --- Validation Phase ---
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        with tqdm(test_loader, desc=f"Epoch {epoch+1}/{EPOCHS} - Validation", unit="batch") as vepoch:
            for inputs, labels in vepoch:
                inputs, labels = inputs.to(device), labels.to(device)
                with torch.amp.autocast('cuda'):
                    outputs = model(inputs)
                    if torch.isnan(outputs).any() or torch.isinf(outputs).any():
                        print("NaN or Inf in model outputs!")
                    if torch.isnan(labels).any() or torch.isinf(labels).any():
                        print("NaN or Inf in labels!")
                    if labels.sum() == 0:
                        print("Skipping batch with all empty masks")
                        continue
                    loss = loss_fn(outputs, labels)
                    preds = (torch.sigmoid(outputs) > 0.5).float()
                val_correct += (preds == labels).sum().item()
                val_total += labels.numel()
                val_acc = val_correct / val_total if val_total > 0 else 0
                val_loss += loss.item()
                vepoch.set_postfix(loss=val_loss / (len(vepoch)))

    avg_val_loss = val_loss / len(test_loader)
    val_losses.append(avg_val_loss)

    print(f"Epoch {epoch+1} - Val Loss: {avg_val_loss:.4f} - Val Acc: {val_acc:.4f}%")

    # --- Scheduler step (use validation loss) ---
    lr_scheduler.step(avg_val_loss)

    # --- Checkpointing ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), checkpoint_path)
        print(f"✅ Model improved (Val Loss={avg_val_loss:.3f}), saved to {checkpoint_path}")
    else:
        patience_counter += 1
        print(f"🔴 No improvement, patience counter: {patience_counter}")

    if patience_counter >= early_stopping_patience:
        print("🛑 Early stopping triggered!")
        break

Epoch 1/200 - Validation: 100%|██████████| 117/117 [00:20<00:00,  5.71batch/s, loss=0.169]


Epoch 1 - Val Loss: 0.1686 - Val Acc: 0.9384%
✅ Model improved (Val Loss=0.169), saved to ../models/corn_SegFormer_date6_seg.pt


Epoch 2/200 - Validation: 100%|██████████| 117/117 [00:13<00:00,  8.72batch/s, loss=0.132]


Epoch 2 - Val Loss: 0.1321 - Val Acc: 0.9507%
✅ Model improved (Val Loss=0.132), saved to ../models/corn_SegFormer_date6_seg.pt


Epoch 3/200 - Validation: 100%|██████████| 117/117 [00:15<00:00,  7.65batch/s, loss=0.112] 


...

Epoch 22 - Val Loss: 0.0770 - Val Acc: 0.9765%
🔴 No improvement, patience counter: 3


Epoch 23/200 - Validation: 100%|██████████| 117/117 [00:16<00:00,  7.06batch/s, loss=0.0845]


Epoch 23 - Val Loss: 0.0845 - Val Acc: 0.9766%
🔴 No improvement, patience counter: 4


Epoch 24/200 - Validation: 100%|██████████| 117/117 [00:15<00:00,  7.34batch/s, loss=0.0786]

Epoch 24 - Val Loss: 0.0786 - Val Acc: 0.9780%
🔴 No improvement, patience counter: 5
🛑 Early stopping triggered!
